In [1]:
import pandas as pd
import json

In [2]:
with open('../../documentation/alpaca_news.csv', 'r') as file:
    news_df = pd.read_csv(file).rename(columns={'symbols': 'tickers'})
    news_list = news_df.to_dict(orient='records')
    json.dumps(news_list, indent=2)

In [12]:
type(news_list[1]['summary'])

float

In [4]:
import ollama

In [5]:
#import ollama
from pydantic import BaseModel, Field
import json

# 1. Define the desired structured output schema
class SentimentAnalysis(BaseModel):
    score: float = Field(
        ..., 
        description="A rating from -1.0 (most negative) to +1.0 (most positive). Neutral is 0.0."
    )

class BatchSentimentAnalysis(BaseModel):
    scores: list[float] = Field(
        ..., description="One sentiment score for each headline, in input order."
    )

class HeadlineSentimentAnalyzer:
    def __init__(
        self,
        model: str = 'qwen3:8b',
        keep_alive: str = '10m',
        system_prompt_in_model: bool = False,
    ):
        self.model = model
        self.keep_alive = keep_alive
        self.system_prompt_in_model = system_prompt_in_model
        self.system_prompt = (
            "You are an expert financial analyst. Your task is to look at a news headline "
            "and judge its sentiment regarding the mentioned publicly traded companies. "
            "Provide a rating score strictly between -1.0 (very bearish/negative) and "
            "+1.0 (very bullish/positive). Highly neutral or purely factual announcements "
            "without clear financial impact should be given a 0.0."
        )

    def analyze_headline_sentiment(self, news_data: dict) -> float:
        """Return the sentiment score for one news dictionary."""
        tickers = ", ".join(news_data.get('tickers', []))
        headline = news_data.get('headline', '')
        user_content = f"Tickers: {tickers}\nHeadline: {headline}"
        messages = [{'role': 'user', 'content': user_content}]
        if not self.system_prompt_in_model:
            messages.insert(0, {'role': 'system', 'content': self.system_prompt})

        try:
            response = ollama.chat(
                model=self.model,
                messages=messages,
                format=SentimentAnalysis.model_json_schema(),
                think=False,
                options={
                    'temperature': 0.0
                },
                keep_alive=self.keep_alive,
            )

            content = response.get('message', {}).get('content', '').strip()
            if not content:
                raise RuntimeError(f"Ollama returned empty content: {response!r}")

            result = SentimentAnalysis.model_validate_json(content)
            return max(-1.0, min(1.0, result.score))

        except ollama.ResponseError as e:
            raise RuntimeError(f"Ollama API error: {e.error}") from e

    def analyze_headlines(
        self, news_items: list[dict], batch_size: int = 5
    ) -> list[float]:
        """Analyze headlines in smaller batches to preserve output alignment."""
        if batch_size < 1:
            raise ValueError('batch_size must be at least 1')
        return [
            score
            for start in range(0, len(news_items), batch_size)
            for score in self._analyze_headline_batch(
                news_items[start:start + batch_size]
            )
        ]

    def _analyze_headline_batch(self, news_items: list[dict]) -> list[float]:
        """Analyze one bounded batch in a single Ollama request."""
        if not news_items:
            return []

        headlines = [
            {
                'tickers': news.get('tickers', []),
                'headline': news.get('headline', '')
            }
            for news in news_items
        ]
        user_content = (
            f"Return exactly {len(headlines)} scores, one for each item, "
            "preserving input order:\n"
            f"{json.dumps(headlines)}"
        )
        messages = [{'role': 'user', 'content': user_content}]
        if not self.system_prompt_in_model:
            messages.insert(0, {'role': 'system', 'content': self.system_prompt})

        try:
            response = ollama.chat(
                model=self.model,
                messages=messages,
                format=BatchSentimentAnalysis.model_json_schema(),
                think=False,
                options={'temperature': 0.0},
                keep_alive=self.keep_alive,
            )
            content = response.get('message', {}).get('content', '').strip()
            if not content:
                raise RuntimeError(f"Ollama returned empty content: {response!r}")

            result = BatchSentimentAnalysis.model_validate_json(content)
            if len(result.scores) != len(news_items):
                raise RuntimeError(
                    f"Expected {len(news_items)} scores, received {len(result.scores)}"
                )
            return [max(-1.0, min(1.0, score)) for score in result.scores]

        except ollama.ResponseError as e:
            raise RuntimeError(f"Ollama API error: {e.error}") from e


In [6]:

news_items = [
    {
        'tickers': ['BUKS', 'HONA'],
        'headline': 'Avcon Industries Receives FAA Supplemental Type Certificate Approval To Install Honeywell Aerospace MKV-A Enhanced Ground Proximity Warning System'
    },
    {
        'tickers': ['AAPL'],
        'headline': 'Apple Reports Strong Quarterly Revenue Growth'
    }
]

analyzer = HeadlineSentimentAnalyzer()
sentiment_scores = analyzer.analyze_headlines(news_items)
for news, sentiment_score in zip(news_items, sentiment_scores):
    print(f"Headline: {news['headline']}")
    print(f"Sentiment Score: {sentiment_score}")

Headline: Avcon Industries Receives FAA Supplemental Type Certificate Approval To Install Honeywell Aerospace MKV-A Enhanced Ground Proximity Warning System
Sentiment Score: 0.0
Headline: Apple Reports Strong Quarterly Revenue Growth
Sentiment Score: 0.5


In [7]:
analyzer = HeadlineSentimentAnalyzer()
sentiment_scores = analyzer.analyze_headlines(news_list[:20])
for news, sentiment_score in zip(news_list[:20], sentiment_scores):
    print(f"Headline: {news['headline']}")
    print(f"Sentiment Score: {sentiment_score}")

Headline: Applied Materials Stock Sinks for 6th Day, but Cantor Sees 85% Upside
Sentiment Score: -0.8
Headline: DOJ Says Saudi National Arrested For Installing Surreptitious Audio Recording Device At OpenAI; An Employee Of OpenAI Has Been Arrested For And Charged With Using Or Endeavoring To Use An Electronic, Mechanical, Or Other Device To Intercept Oral Communications
Sentiment Score: -0.2
Headline: Avcon Industries Receives FAA Supplemental Type Certificate Approval To Install Honeywell Aerospace MKV-A Enhanced Ground Proximity Warning System
Sentiment Score: 0.0
Headline: U.S. Department Of The Treasury Announced Enforcement Actions Imposing More Than $1.21B In Penalties Against Bank Of America
Sentiment Score: -0.9
Headline: Dan Ives Says AI Stock Selloff ‘Opportunity to Own the Winners’: Here Are 7 Companies He Named
Sentiment Score: 0.6
Headline: Epic Games’ Disney Game Faces Delays and An AI Overhaul After $1.5 Billion Investment
Sentiment Score: -0.6
Headline: $100 Invested In

In [8]:
analyzer = HeadlineSentimentAnalyzer()
sentiment_scores = analyzer.analyze_headlines(news_list[:100])
for news, sentiment_score in zip(news_list[:100], sentiment_scores):
    print(f"Headline: {news['headline']}")
    print(f"Sentiment Score: {sentiment_score}")

Headline: Applied Materials Stock Sinks for 6th Day, but Cantor Sees 85% Upside
Sentiment Score: -0.8
Headline: DOJ Says Saudi National Arrested For Installing Surreptitious Audio Recording Device At OpenAI; An Employee Of OpenAI Has Been Arrested For And Charged With Using Or Endeavoring To Use An Electronic, Mechanical, Or Other Device To Intercept Oral Communications
Sentiment Score: -0.2
Headline: Avcon Industries Receives FAA Supplemental Type Certificate Approval To Install Honeywell Aerospace MKV-A Enhanced Ground Proximity Warning System
Sentiment Score: 0.0
Headline: U.S. Department Of The Treasury Announced Enforcement Actions Imposing More Than $1.21B In Penalties Against Bank Of America
Sentiment Score: -0.9
Headline: Dan Ives Says AI Stock Selloff ‘Opportunity to Own the Winners’: Here Are 7 Companies He Named
Sentiment Score: 0.6
Headline: Epic Games’ Disney Game Faces Delays and An AI Overhaul After $1.5 Billion Investment
Sentiment Score: -0.3
Headline: $100 Invested In

In [9]:
news_list[0]

{'id': 62341492,
 'created_at': '2026-10-13T20:51:37Z',
 'headline': 'Applied Materials Stock Sinks for 6th Day, but Cantor Sees 85% Upside',
 'author': 'Piero Cingari',
 'tickers': "['AMAT', 'ASML', 'AVGO', 'KLAC', 'LRCX', 'MU', 'SKHY', 'SSNLF']",
 'source': 'benzinga',
 'summary': 'Applied Materials stock fell 7% on investor day, its sixth straight loss. Cantor Fitzgerald kept an $850 target, implying 85% upside.',
 'url': 'https://www.benzinga.com/analyst-stock-ratings/analyst-color/26/10/62341492/applied-materials-stock-sinks-sixth-day-cantor-850-target'}

In [10]:
analyzer = HeadlineSentimentAnalyzer()
sentiment_scores = analyzer.analyze_headlines(news_list)
for news, sentiment_score in zip(news_list, sentiment_scores):
    print(f"ticker: {news['tickers']}", f", -Sentiment Score: {sentiment_score}")
    print(f"Headline: {news['headline']}")
    

ticker: ['AMAT', 'ASML', 'AVGO', 'KLAC', 'LRCX', 'MU', 'SKHY', 'SSNLF'] , -Sentiment Score: -0.8
Headline: Applied Materials Stock Sinks for 6th Day, but Cantor Sees 85% Upside
ticker: ['MSFT'] , -Sentiment Score: -0.2
Headline: DOJ Says Saudi National Arrested For Installing Surreptitious Audio Recording Device At OpenAI; An Employee Of OpenAI Has Been Arrested For And Charged With Using Or Endeavoring To Use An Electronic, Mechanical, Or Other Device To Intercept Oral Communications
ticker: ['BUKS', 'HONA'] , -Sentiment Score: 0.0
Headline: Avcon Industries Receives FAA Supplemental Type Certificate Approval To Install Honeywell Aerospace MKV-A Enhanced Ground Proximity Warning System
ticker: ['BAC'] , -Sentiment Score: -0.9
Headline: U.S. Department Of The Treasury Announced Enforcement Actions Imposing More Than $1.21B In Penalties Against Bank Of America
ticker: ['CRM', 'CRWD', 'META', 'MSFT', 'NOW', 'NVDA', 'PLTR'] , -Sentiment Score: 0.6
Headline: Dan Ives Says AI Stock Selloff 